#### Package 설치

In [ ]:
# uv add docx2txt
# uv add langchain-upstage

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[:5])

up_gq


#### 1. Knowledge Base 구성을 위한 데이터 생성

- [RecursiveCharacterTextSplitter](https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/)를 활용한 데이터 chunking
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
    - LangChain에서 다양한 [TextSplitter](https://python.langchain.com/v0.2/docs/how_to/#text-splitters)들을 제공
- `chunk_size` 는 split 된 chunk의 최대 크기
- `chunk_overlap`은 앞 뒤로 나뉘어진 chunk들이 얼마나 겹쳐도 되는지 지정

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('../data/tax_with_table.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

print(len(document_list))
print(type(document_list[0]))
print(document_list[:2])

c:\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


225
<class 'langchain_core.documents.base.Document'>
[Document(metadata={'source': '../data/tax_with_table.docx'}, page_content='소득세법\n\n소득세법\n\n[시행 2024. 1. 1.] [법률 제19933호, 2023. 12. 31., 일부개정]\n\n기획재정부(소득세제과(사업소득, 기타소득)) 044-215-4217\n\n기획재정부(소득세제과(근로소득)) 044-215-4216\n\n기획재정부(재산세제과(양도소득세)) 044-215-4314\n\n기획재정부(금융세제과(이자소득, 배당소득)) 044-215-4236\n\n\n\n\t제1장 총칙 <개정 2009. 12. 31.>\t\n\n\n\n제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다.\n\n[본조신설 2009. 12. 31.]\n\n[종전 제1조는 제2조로 이동 <2009. 12. 31.>]\n\n\n\n제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 12. 31.>\n\n1. “거주자”란 국내에 주소를 두거나 183일 이상의 거소(居所)를 둔 개인을 말한다.\n\n2. “비거주자”란 거주자가 아닌 개인을 말한다.\n\n3. “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다.\n\n4. “외국법인”이란 「법인세법」 제2조제3호에 따른 외국법인을 말한다.\n\n5. “사업자”란 사업소득이 있는 거주자를 말한다.\n\n② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다.\n\n[본조신설 2009. 12. 31.]\n\n\n\n제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무

In [3]:
from langchain_upstage import UpstageEmbeddings

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
print(embeddings)

client=<openai.resources.embeddings.Embeddings object at 0x000001672DC7F5F0> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001672D4A31D0> model='solar-embedding-1-large' dimensions=None upstage_api_key=SecretStr('**********') upstage_api_base='https://api.upstage.ai/v1/solar' embedding_ctx_length=4096 embed_batch_size=10 allowed_special=set() disallowed_special='all' chunk_size=1000 max_retries=2 request_timeout=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers={'x-upstage-client': 'langchain'} default_query=None http_client=None http_async_client=None


In [4]:
from langchain_community.vectorstores import FAISS

# 데이터를 처음 저장할 때 
database = FAISS.from_documents(documents=document_list, embedding=embeddings)
# 로컬 파일로 저장
database.save_local("faiss_docdb")

# 이미 저장된 데이터를 사용할 때 
# database = FAISS.load_local("faiss_docdb", embedding, allow_dangerous_deserialization=True)
print(database)

#### 2. 답변 생성을 위한 Retrieval

- `FAISS`에 저장한 데이터를 유사도 검색(`similarity_search()`)를 활용해서 가져옴

In [5]:
query = '비과세소득에는 어떤것들이 있나요?'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정
retrieved_docs = database.similarity_search(query, k=6)

print(len(retrieved_docs))
print(type(retrieved_docs[0]))
print(retrieved_docs[0].metadata)

6
<class 'langchain_core.documents.base.Document'>
{'source': '../data/tax_with_table.docx'}


In [6]:
print(retrieved_docs[0].page_content[:100])

마. 「국군포로의 송환 및 대우 등에 관한 법률」에 따라 국군포로가 받는 위로지원금과 그 밖의 금품

바. 「문화재보호법」에 따라 국가지정문화재로 지정된 서화ㆍ골동품의 양도로 발생


#### 3. Augmentation을 위한 Prompt 활용

- Retrieval된 데이터는 LangChain에서 제공하는 프롬프트(`"rlm/rag-prompt"`) 사용

In [7]:

from langchain_upstage import ChatUpstage

print("===> 6. 생성 → LLM으로 답변 생성")
llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm)

===> 6. 생성 → LLM으로 답변 생성
profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000001672C757C80> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001673122EF90> root_client=<openai.OpenAI object at 0x000001672C64C3B0> root_async_client=<openai.AsyncOpenAI object at 0x000001673122F380> model_name='solar-pro' temperature=0.5 model_kwargs={} disabled_params={'parallel_tool_calls': None} upstage_api_key=SecretStr('**********') upstage_api_base='https://api.upstage.ai/v1'


In [11]:
try:
    from langchain_classic import hub
    rag_prompt = hub.pull("rlm/rag-prompt")
    print(" langchain-classic hub 작동")
except ImportError:
    print(" langchain-classic 미설치")

 langchain-classic hub 작동


#### 4. 답변 생성

- [RetrievalQA](https://docs.smith.langchain.com/old/cookbook/hub-examples/retrieval-qa-chain)를 통해 LLM에 전달
    - `RetrievalQA`는 [create_retrieval_chain](https://python.langchain.com/v0.2/docs/how_to/qa_sources/#using-create_retrieval_chain)으로 대체됨
    - 실제 ChatBot 구현 시 `create_retrieval_chain`으로 변경하는 과정을 볼 수 있음

In [ ]:
from langchain_classic.chains import RetrievalQA  # langchain-classic 사용
from pprint import pprint

# 기존 코드 그대로 작동 (레거시 지원)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm, 
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": rag_prompt}
)

In [13]:
query = '비과세소득에는 어떤것들이 있나요?'

ai_message = qa_chain.invoke({"query": query})
pprint(ai_message)

{'query': '비과세소득에는 어떤것들이 있나요?',
 'result': '비과세소득에는 국군포로 위로지원금, 국가지정문화재 서화·골동품 양도 소득, 박물관·미술관 양도 소득, 종교인소득 중 '
           '학자금·식사대·실비변상적 지급액(월 20만원 이내 보육금·사택 이익), 위원회 수당 등이 포함됩니다. 또한, 공익신탁 '
           '이익, 주택임대소득(일정 조건), 농어가부업소득, 학자금·실비변상적 급여 등도 비과세 대상입니다. 자세한 항목은 관련 '
           '법령 제12조를 참조하시기 바랍니다.'}


In [14]:
query = '과세소득의 범위 및 소득의 구분에는 어떤것들이 있나요?'

ai_message = qa_chain.invoke({"query": query})
pprint(ai_message)

{'query': '과세소득의 범위 및 소득의 구분에는 어떤것들이 있나요?',
 'result': '과세소득의 범위는 거주자의 경우 모든 소득(제3조 ①), 비거주자의 경우 국내원천소득(제3조 ②)에 대해 과세됩니다. '
           '소득 구분은 거주자의 경우 종합소득(이자·배당·사업·근로·연금·기타소득), 퇴직소득, 금융투자소득, 양도소득으로 '
           '나뉘며(제4조 ①), 비거주자는 국내원천소득에 따라 구분됩니다(제4조 ③). 신탁재산 소득은 수익자 또는 위탁자에게 '
           '귀속됩니다(제2조의3).'}


* LangChain 기반의 RAG(Retrieval-Augmented Generation) 파이프라인을 구현하여 DOCX 문서를 로드, 임베딩, 검색, 그리고 LLM을 통한 답변 생성

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter  # LCEL 호환
from langchain_core.documents import Document  #  최신 import
from langchain_core.runnables import RunnablePassthrough  #  LCEL
from langchain_core.output_parsers import StrOutputParser  #  LCEL
from langchain_core.prompts import PromptTemplate  #  LCEL
from langchain_community.document_loaders import Docx2txtLoader
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_classic import hub  #  hub는 classic으로

import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# 1. 환경 변수 로드
load_dotenv()

# 2. API 키 확인
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
if not UPSTAGE_API_KEY:
    raise ValueError("UPSTAGE_API_KEY 설정 필요 (.env)")

# 3. DOCX 로드 함수
def load_docx(file_path):
    loader = Docx2txtLoader(file_path)
    documents = loader.load()
    text = "\n".join([doc.page_content for doc in documents])
    if not text.strip():
        raise ValueError("문서 텍스트 없음")
    return text

# 4. 문서 분할
def split_text(text, chunk_size=500, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    return splitter.split_text(text)

# 5. 벡터스토어 생성
def create_vector_store(text_chunks, embedding_model):
    documents = [Document(page_content=chunk) for chunk in text_chunks]
    vector_store = FAISS.from_documents(documents, embedding_model)
    vector_store.save_local("faiss_docdb")
    return vector_store

# 6. LCEL RAG 체인 (RetrievalQA 대체!)
def create_rag_chain(llm, retriever, prompt):
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

# 7. 질문 응답 함수
def query_with_llm(query, vector_store):
    llm = ChatUpstage(model="solar-pro", temperature=0.5)
    
    # Hub 프롬프트 (classic 사용)
    rag_prompt = hub.pull("rlm/rag-prompt")
    
    retriever = vector_store.as_retriever(search_kwargs={"k": 6})
    rag_chain = create_rag_chain(llm, retriever, rag_prompt)
    
    answer = rag_chain.invoke(query)
    return answer

# 실행
if __name__ == "__main__":
    docx_path = "../data/tax_with_table.docx"
    
    # 1. 문서 로드
    text = load_docx(docx_path)
    print(" 문서 로드 완료")
    
    # 2. 분할
    text_chunks = split_text(text)
    print(f" {len(text_chunks)}개 청크 생성")
    
    # 3. 임베딩
    embedding_model = UpstageEmbeddings(model="solar-embedding-1-large")
    
    # 4. 벡터스토어
    vector_store = create_vector_store(text_chunks, embedding_model)
    print(" 벡터스토어 생성 완료")
    
    # 5. 질의
    query = "총수입금액 불산입에 대하여 설명해 주세요."
    answer = query_with_llm(query, vector_store)
    
    print("\n AI 답변:")
    print(answer)

 문서 로드 완료
 720개 청크 생성
 벡터스토어 생성 완료

 AI 답변:
총수입금액 불산입은 소득세 계산 시 특정 항목을 총수입금액에 포함하지 않는 것을 의미합니다. 이는 개별소비세·주세(매입세액 제외), 국세·지방세 환급가산금, 부가가치세 매출세액, 석유판매업자 환급세액 등이 해당됩니다. 또한, 무상위 자산가액 중 이월결손금 보전액 등도 불산입됩니다. (참고: 제26조, 제⑦~⑩항 등)


In [16]:
from pprint import pprint

pprint(answer)

('총수입금액 불산입은 소득세 계산 시 특정 항목을 총수입금액에 포함하지 않는 것을 의미합니다. 이는 개별소비세·주세(매입세액 제외), '
 '국세·지방세 환급가산금, 부가가치세 매출세액, 석유판매업자 환급세액 등이 해당됩니다. 또한, 무상위 자산가액 중 이월결손금 보전액 등도 '
 '불산입됩니다. (참고: 제26조, 제⑦~⑩항 등)')


###  개선된 Source Level1

* uv add unstructured[docx]

In [20]:
import os
import re
from dotenv import load_dotenv
from typing import List
import warnings

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredWordDocumentLoader  # Docx2txtLoader 대체
from langchain_core.prompts import ChatPromptTemplate  # PromptTemplate → ChatPromptTemplate
from langchain_upstage import UpstageEmbeddings, ChatUpstage

warnings.filterwarnings("ignore", category=UserWarning)

# 1. 환경 변수 로드
load_dotenv()

# 2. Upstage API 키 확인
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
if not UPSTAGE_API_KEY:
    raise ValueError("UPSTAGE API 키가 설정되지 않았습니다. .env 파일을 확인하세요.")

# 2. 한국어 법률 문서 전용 텍스트 전처리 함수 (변경 없음)
def preprocess_korean_legal_text(text: str) -> str:
    """한국어 법률 문서를 위한 전처리."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'제(\d+)조', r'제\1조', text)
    text = re.sub(r'①|②|③|④|⑤|⑥|⑦|⑧|⑨|⑩', 
                  lambda m: f"제{ord(m.group()) - ord('①') + 1}항", text)
    text = re.sub(r'(\d+)\.\s', r'제\1호 ', text)
    return text.strip()

# 3. 개선된 문서 분할 함수 (최신 splitter 사용)
def advanced_split_text(text: str, chunk_size: int = 800, chunk_overlap: int = 200) -> List[str]:
    """법률 문서에 최적화된 텍스트 분할."""
    text = preprocess_korean_legal_text(text)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=[
            "\n제", "\n**제", "\n①", "\n②", "\n③", "\n④", "\n⑤",
            "\n1.", "\n2.", "\n3.", "\n4.", "\n5.",
            "\n가.", "\n나.", "\n다.", "\n라.", "\n마.",
            "\n\n", "\n", ". ", " ", ""
        ]
    )
    return splitter.split_text(text)

# 4. 개선된 문서 로더 (UnstructuredWordDocumentLoader 사용)
def load_docx_advanced(file_path: str) -> str:
    """개선된 DOCX 파일 로더."""
    try:
        loader = UnstructuredWordDocumentLoader(file_path)
        documents = loader.load()
        text = "\n".join([doc.page_content for doc in documents])
        if not text.strip():
            raise ValueError("문서에서 텍스트를 추출할 수 없습니다.")
        text = re.sub(r'\n\s*\n', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        return text
    except Exception as e:
        raise RuntimeError(f"문서 로딩 실패: {str(e)}")

# 5. 벡터 저장소 생성 함수 (FAISS.from_documents 유지 - 여전히 지원됨)
def create_vector_store(text_chunks: List[str], embedding_model) -> tuple:
    """메타데이터가 포함된 벡터 저장소 생성."""
    try:
        documents: List[Document] = []
        for i, chunk in enumerate(text_chunks):
            metadata = {
                'chunk_id': i,
                'chunk_length': len(chunk),
                'chunk_type': 'legal_document'
            }
            if '제' in chunk and '조' in chunk:
                article_match = re.search(r'제(\d+)조', chunk)
                if article_match:
                    metadata['article'] = f"제{article_match.group(1)}조"
            documents.append(Document(page_content=chunk, metadata=metadata))
        
        vector_store = FAISS.from_documents(documents, embedding_model)
        vector_store.save_local("faiss_docdb")
        return vector_store, documents
    except Exception as e:
        raise RuntimeError(f"벡터 저장소 생성 실패: {str(e)}")

# 6. 키워드 기반 검색 함수 (변경 없음)
def keyword_search(query: str, documents: List[Document], k: int = 5) -> List[Document]:
    """간단한 키워드 기반 검색."""
    query_words = set(query.lower().split())
    scores = []
    for i, doc in enumerate(documents):
        content_words = set(doc.page_content.lower().split())
        intersection = query_words.intersection(content_words)
        score = len(intersection) / len(query_words) if query_words else 0
        if query.lower() in doc.page_content.lower():
            score += 0.5
        scores.append((score, i, doc))
    scores.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, _, doc in scores[:k]]

# 7. 하이브리드 검색 함수 (변경 없음)
def hybrid_search(query: str, vector_store, documents: List[Document], k: int = 5, alpha: float = 0.7) -> List[Document]:
    """벡터 검색과 키워드 검색을 결합한 하이브리드 검색."""
    vector_results = vector_store.similarity_search(query, k=k*2)
    keyword_results = keyword_search(query, documents, k=k*2)
    combined_results = {}
    
    for i, doc in enumerate(vector_results):
        doc_id = id(doc)  # page_content 대신 id 사용 (해시 충돌 방지)
        vector_score = alpha * (1.0 - i / len(vector_results))
        combined_results[doc_id] = {'document': doc, 'score': vector_score, 'vector_rank': i + 1}
    
    for i, doc in enumerate(keyword_results):
        doc_id = id(doc)
        keyword_score = (1 - alpha) * (1.0 - i / len(keyword_results))
        if doc_id in combined_results:
            combined_results[doc_id]['score'] += keyword_score
            combined_results[doc_id]['keyword_rank'] = i + 1
        else:
            combined_results[doc_id] = {'document': doc, 'score': keyword_score, 'keyword_rank': i + 1}
    
    sorted_results = sorted(combined_results.values(), key=lambda x: x['score'], reverse=True)
    return [result['document'] for result in sorted_results[:k]]

# 8. 한국어 법률 문서 전용 프롬프트 (ChatPromptTemplate 사용)
def create_korean_legal_prompt() -> ChatPromptTemplate:
    """한국어 법률 문서용 프롬프트 템플릿."""
    template = """당신은 한국 세법 전문가입니다. 주어진 법률 문서를 바탕으로 정확하고 자세한 답변을 제공해야 합니다.

다음 규칙을 반드시 따르세요:
1. 법조문의 조항, 항, 호, 목을 정확히 인용하세요
2. 전문 용어를 사용할 때는 쉬운 설명을 함께 제공하세요
3. 관련 조항들 간의 연관성을 설명하세요
4. 실무적 적용 방법도 함께 설명하세요
5. 불확실한 내용이 있으면 명시적으로 언급하세요

참고 문서:
{context}

질문: {question}

위 법률 문서를 바탕으로 정확하고 자세한 답변을 제공해주세요. 관련 조항을 인용하며 설명해주세요."""
    return ChatPromptTemplate.from_template(template)

# 9. 질문 응답 함수 (llm.invoke([HumanMessage]) 방식 지원)
def query_with_llm(query: str, vector_store, documents: List[Document]):
    """개선된 LLM 기반 질문 응답."""
    try:
        llm = ChatUpstage(
            model="solar-pro",
            temperature=0.5  # base_url 제거 - 기본값 사용
        )
        
        print(f"질의: {query}")
        print("하이브리드 검색 수행 중...")
        
        relevant_docs = hybrid_search(query, vector_store, documents, k=7, alpha=0.7)
        print(f"검색된 관련 문서: {len(relevant_docs)}개")
        
        context = "\n\n".join([f"[문서 {i+1}]\n{doc.page_content}" for i, doc in enumerate(relevant_docs)])
        prompt = create_korean_legal_prompt()
        
        print("LLM 응답 생성 중...")
        response = llm.invoke(prompt.format_messages(context=context, question=query))  # format_messages 사용
        
        return {
            "answer": response.content,
            "source_documents": relevant_docs,
            "context_used": context
        }
    except Exception as e:
        raise RuntimeError(f"LLM 응답 생성 실패: {str(e)}")

# 10. 컨텍스트 품질 평가 함수 (변경 없음)
def evaluate_context_quality(query: str, retrieved_docs: List[Document]) -> float:
    """검색된 문서의 품질을 간단히 평가."""
    query_words = set(query.lower().split())
    quality_scores = []
    for doc in retrieved_docs:
        doc_words = set(doc.page_content.lower().split())
        keyword_match = len(query_words.intersection(doc_words)) / len(query_words)
        length_score = min(len(doc.page_content) / 1000, 1.0)
        total_score = (keyword_match * 0.7) + (length_score * 0.3)
        quality_scores.append(total_score)
    avg_quality = sum(quality_scores) / len(quality_scores) if quality_scores else 0
    return avg_quality

# 실행 예제
if __name__ == "__main__":
    docx_path = "../data/tax_with_table.docx"
    
    print("=== 개선된 RAG 파이프라인 실행 ===\n")
    
    # 1. 문서 로드
    print("1. 문서 로드 중...")
    text = load_docx_advanced(docx_path)
    print(f"   문서 로드 완료: {len(text):,} 문자\n")
    
    # 2. 개선된 문서 분할
    print("2. 문서 분할 중...")
    text_chunks = advanced_split_text(text, chunk_size=800, chunk_overlap=200)
    print(f"   문서 분할 완료: {len(text_chunks)}개 청크 생성\n")
    
    # 3. 임베딩 모델 초기화
    print("3. 임베딩 모델 초기화...")
    embedding_model = UpstageEmbeddings(model="solar-embedding-1-large")
    print("   임베딩 모델 초기화 완료\n")
    
    # 4. 벡터 저장소 생성
    print("4. 벡터 저장소 생성 중...")
    vector_store, documents = create_vector_store(text_chunks, embedding_model)
    print("   벡터 저장소 생성 완료\n")
    
    # 5. 질의 실행
    print("5. 질의 실행 중...")
    query = "총수입금액 불산입에 대하여 설명해 주세요."
    results = query_with_llm(query, vector_store, documents)
    print("\n답변:")
    print(results["answer"])

=== 개선된 RAG 파이프라인 실행 ===

1. 문서 로드 중...
   문서 로드 완료: 289,275 문자

2. 문서 분할 중...
   문서 분할 완료: 511개 청크 생성

3. 임베딩 모델 초기화...
   임베딩 모델 초기화 완료

4. 벡터 저장소 생성 중...
   벡터 저장소 생성 완료

5. 질의 실행 중...
질의: 총수입금액 불산입에 대하여 설명해 주세요.
하이브리드 검색 수행 중...
검색된 관련 문서: 7개
LLM 응답 생성 중...

답변:
### 총수입금액 불산입에 대한 상세 설명

총수입금액 불산입은 소득세 계산 시 특정 금액을 총수입금액에 포함하지 않는 것을 의미합니다. 이는 중복 과세 방지, 실질적 소득 반영, 또는 정책적 목적(예: 세금 환급금, 자산 무상수취 등)을 위해 적용됩니다. 아래는 관련 조항을 중심으로 체계적으로 설명한 내용입니다.

---

#### 1. **세금 환급금 및 충당금 (제26조 제1항, 제8항)**
- **조문 인용**:  
  - **제26조 제1항**: "소득세 또는 개인지방소득세 환급금 중 다른 세액에 충당한 금액은 총수입금액에 산입하지 아니한다."  
  - **제26조 제8항**: "국세/지방세 환급가산금, 과오납금 이자는 총수입금액에 산입하지 아니한다."  
- **설명**:  
  - 환급금은 이미 납부한 세금을 돌려받는 것이므로 소득으로 보지 않습니다.  
  - **연관 조항**: 제26조 제1항과 제8항은 모두 "환급금"을 다루며, 중복 과세 방지를 목적으로 합니다.  
- **실무 적용**:  
  - 예: A씨가 2023년 소득세 500만 원을 환급받고, 이를 지방세 납부에 사용한 경우, 500만 원은 총수입금액에 포함되지 않습니다.

---

#### 2. **무상 자산 수취 및 채무 면제 (제26조 제2항)**
- **조문 인용**:  
  - **제26조 제2항**: "무상으로 받은 자산 가액(복식부기의무자의 국고보조금 등 제외)과 채무 면제로 인한 부채 감소액 중 

In [21]:
from pprint import pprint

pprint(results)

{'answer': '### 총수입금액 불산입에 대한 상세 설명\n'
           '\n'
           '총수입금액 불산입은 소득세 계산 시 특정 금액을 총수입금액에 포함하지 않는 것을 의미합니다. 이는 중복 과세 방지, '
           '실질적 소득 반영, 또는 정책적 목적(예: 세금 환급금, 자산 무상수취 등)을 위해 적용됩니다. 아래는 관련 조항을 '
           '중심으로 체계적으로 설명한 내용입니다.\n'
           '\n'
           '---\n'
           '\n'
           '#### 1. **세금 환급금 및 충당금 (제26조 제1항, 제8항)**\n'
           '- **조문 인용**:  \n'
           '  - **제26조 제1항**: "소득세 또는 개인지방소득세 환급금 중 다른 세액에 충당한 금액은 총수입금액에 산입하지 '
           '아니한다."  \n'
           '  - **제26조 제8항**: "국세/지방세 환급가산금, 과오납금 이자는 총수입금액에 산입하지 아니한다."  \n'
           '- **설명**:  \n'
           '  - 환급금은 이미 납부한 세금을 돌려받는 것이므로 소득으로 보지 않습니다.  \n'
           '  - **연관 조항**: 제26조 제1항과 제8항은 모두 "환급금"을 다루며, 중복 과세 방지를 목적으로 '
           '합니다.  \n'
           '- **실무 적용**:  \n'
           '  - 예: A씨가 2023년 소득세 500만 원을 환급받고, 이를 지방세 납부에 사용한 경우, 500만 원은 '
           '총수입금액에 포함되지 않습니다.\n'
           '\n'
           '---\n'
           '\n'
           '#### 2. **무상 자산 수취 및 채무 면제 (제26조 제2항)**\n'
 

In [22]:
# 6. 컨텍스트 품질 평가
context_quality = evaluate_context_quality(query, results["source_documents"])

# 7. 결과 출력
print("\n" + "="*60)
print(" AI의 답변:")
print("="*60)
print(results["answer"])

print("\n" + "="*60)
print(" 검색 결과 요약:")
print("="*60)
print(f"• 참고한 문서 조각 수: {len(results['source_documents'])}개")
print(f"• 컨텍스트 품질 점수: {context_quality:.2f}/1.00")
print(f"• 총 컨텍스트 길이: {len(results['context_used']):,} 문자")

print("\n" + "="*60)
print("📄 참고한 문서 미리보기:")
print("="*60)
for i, doc in enumerate(results["source_documents"][:3]):  # 상위 3개만 미리보기
    preview = doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content
    print(f"\n[문서 {i+1}] {preview}")
print("="*60)


 AI의 답변:
### 총수입금액 불산입에 대한 상세 설명

총수입금액 불산입은 소득세 계산 시 특정 금액을 총수입금액에 포함하지 않는 것을 의미합니다. 이는 중복 과세 방지, 실질적 소득 반영, 또는 정책적 목적(예: 세금 환급금, 자산 무상수취 등)을 위해 적용됩니다. 아래는 관련 조항을 중심으로 체계적으로 설명한 내용입니다.

---

#### 1. **세금 환급금 및 충당금 (제26조 제1항, 제8항)**
- **조문 인용**:  
  - **제26조 제1항**: "소득세 또는 개인지방소득세 환급금 중 다른 세액에 충당한 금액은 총수입금액에 산입하지 아니한다."  
  - **제26조 제8항**: "국세/지방세 환급가산금, 과오납금 이자는 총수입금액에 산입하지 아니한다."  
- **설명**:  
  - 환급금은 이미 납부한 세금을 돌려받는 것이므로 소득으로 보지 않습니다.  
  - **연관 조항**: 제26조 제1항과 제8항은 모두 "환급금"을 다루며, 중복 과세 방지를 목적으로 합니다.  
- **실무 적용**:  
  - 예: A씨가 2023년 소득세 500만 원을 환급받고, 이를 지방세 납부에 사용한 경우, 500만 원은 총수입금액에 포함되지 않습니다.

---

#### 2. **무상 자산 수취 및 채무 면제 (제26조 제2항)**
- **조문 인용**:  
  - **제26조 제2항**: "무상으로 받은 자산 가액(복식부기의무자의 국고보조금 등 제외)과 채무 면제로 인한 부채 감소액 중 이월결손금 보전에 충당된 금액은 총수입금액에 산입하지 아니한다."  
- **설명**:  
  - 무상 자산 수취는 일반적으로 기타소득으로 과세되지만, **이월결손금 보전**에 사용된 경우 예외를 인정합니다.  
  - **예외**: 복식부기의무자가 받은 국고보조금은 제외(제160조 참조).  
- **실무 적용**:  
  - 예: B씨가 채무 3,000만 원을 면제받고, 이를 과거 결손금 2,000만 원 보전에 사용한 경우, 2,000만 원은 총수입금액